# Micrograd and Backprogration

In [ ]:
# import os
# os.environ["PATH"] += os.pathsep + r"C:\Users\user\Downloads\windows_10_cmake_Release_Graphviz-14.1.4-win32\Graphviz-14.1.4-win32\bin"

In [ ]:
import math
import numpy as np

### Main Part

In [ ]:
class Value:
  
  def __init__(self, data, _children=(), _op='', label=''):
    self.data = data
    self.grad = 0.0
    self._backward = lambda: None
    self._prev = set(_children)
    self._op = _op
    self.label = label

  def __repr__(self):
    return f"Value(data={self.data})"
  
  def __add__(self, other):
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data + other.data, (self, other), '+')
    
    def _backward():
      self.grad += 1.0 * out.grad
      other.grad += 1.0 * out.grad
    out._backward = _backward
    
    return out

  def __neg__(self):
    return self * -1

  def __sub__(self, other):
    return self + (-other)

  def __mul__(self, other):
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data * other.data, (self, other), '*')
    
    def _backward():
      self.grad += other.data * out.grad
      other.grad += self.data * out.grad
    out._backward = _backward
      
    return out
  
  def __rmul__(self, other): # Fallback for __mul__()
    return self * other
  
  def __pow__(self, other):
      assert isinstance(other, (int, float)), "Supporting int or float powers"
      out = Value(self.data**other, (self,), f'**{other}')

      def _backward():
          self.grad += (other * self.data**(other-1)) * out.grad
      out._backward = _backward

      return out
  
  def __truediv__(self, other):
    return self * other**-1
  
  def tanh(self):
    x = self.data
    t = (math.exp(2*x) - 1)/(math.exp(2*x) + 1)
    out = Value(t, (self, ), 'tanh')
    
    def _backward():
      self.grad += (1 - t**2) * out.grad
    out._backward = _backward
    
    return out
  
  def exp(self):
    x = self.data
    out = Value(math.exp(x), (self,), 'exp')

    def _backward():
      self.grad += out.data * out.grad
    out._backward = _backward

    return out
  
  def backward(self):
    # Topological sort - goes from Left to Right
    topo = []
    visited = set()
    def build_topo(v):
      if v not in visited:
        visited.add(v)
        for child in v._prev:
          build_topo(child)
        topo.append(v)
    build_topo(self)
    
    self.grad = 1.0
    for node in reversed(topo):
      node._backward()

In [56]:
# Checking the operations of Value class
a = Value(2)
b = Value(4)

print("Add: ",a+b)
print("Subtract: ",a-b)
print("Multiply: ",a*b)
print("Negative: ",-a)
print("Divide: ",a*b**-1)
print("Power: ", a**3)
print("tanh(a): ",Value.tanh(a))

In [51]:
# Making different equations to try
a = Value(2.0, label='a')
b = Value(-3.0, label='b')
c = Value(10.0, label='c')
e = a*b; e.label = 'e'
d = e + c; d.label = 'd'
f = Value(-2.0, label='f')
L = d * f; L.label = 'L'
L

In [4]:
# Tracing nodes,grads through graph
from graphviz import Digraph

def trace(root):
  # builds a set of all nodes and edges in a graph
  nodes, edges = set(), set()
  def build(v):
    if v not in nodes:
      nodes.add(v)
      for child in v._prev:
        edges.add((child, v))
        build(child)
  build(root)
  return nodes, edges

def draw_dot(root):
  dot = Digraph(format='svg', graph_attr={'rankdir': 'LR'}) # LR = left to right
  
  nodes, edges = trace(root)
  for n in nodes:
    uid = str(id(n))
    # for any value in the graph, create a rectangular ('record') node for it
    dot.node(name = uid, label = "{ %s | data %.4f | grad %.4f }" % (n.label, n.data, n.grad), shape='record')
    if n._op:
      # if this value is a result of some operation, create an op node for it
      dot.node(name = uid + n._op, label = n._op)
      # and connect this node to it
      dot.edge(uid + n._op, uid)

  for n1, n2 in edges:
    # connect n1 to the op node of n2
    dot.edge(str(id(n1)), str(id(n2)) + n2._op)

  return dot

In [49]:
# Manually change the value to find the derivative using First principle
def comp_derivative():
  "f'(x) = (f(x+h) - f(x))/h"
  
  h = 0.001
  
  a = Value(2.0, label='a')
  b = Value(-3.0, label='b')
  c = Value(10.0, label='c')
  e = a*b; e.label = 'e'
  d = e + c; d.label = 'd'
  f = Value(-2.0, label='f')
  L = d * f; L.label = 'L'
  L1 = L.data
  
  # Change add 'h' in required variable and compute
  a = Value(2.0, label='a')
  b = Value(-3.0, label='b')
  c = Value(10.0, label='c')
  c.data += h # Gives c.grad
  e = a*b; e.label = 'e'
  d = e + c; d.label = 'd'
  f = Value(-2.0, label='f')
  L = d * f; L.label = 'L'
  L2 = L.data
  
  print((L2 - L1)/h)
  
comp_derivative()

In [53]:
L.grad = 1
d.grad = -2
f.grad = 4
c.grad = -2
e.grad = -2
a.grad = 6
b.grad = -4

draw_dot(L)

- Shows Gradient across all nodes when the variables' values are manually computed

In [ ]:
# Nudging the input to get the Loss function to go as close to 0
a.data += 0.01 * a.grad # Should change in the direction of gradient
b.data += 0.01 * b.grad # a,b,c,f are leaf nodes
c.data += 0.01 * c.grad
f.data += 0.01 * f.grad
e = a * b
d = e + c
L = d * f
L.data

### Backpropagation through neuron

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Visualizing a tanh() trigonometric function
plt.plot(np.arange(-5,5,0.2), np.tanh(np.arange(-5,5,0.2)))

In [59]:
# Inputs
x1 = Value(2.0, label='x1')
x2 = Value(0.0, label='x2')

# Weights
w1 = Value(-3.0, label='w1')
w2 = Value(1.0, label='w2')

# Bias
b = Value(6.88137, label='b')

# Weighted
x1w1 = x1*w1; x1w1.label = 'x1*w1'
x2w2 = x2*w2; x2w2.label = 'x2*w2'

# Weighted sum
x1w1x2w2 = x1w1 + x2w2; x1w1x2w2.label = 'x1*w1 + x2*w2'
n = x1w1x2w2 + b; n.label = 'n'
o = n.tanh(); o.label = 'o'

- Doing the same as before, but trying to find a pattern

In [56]:
o.grad = 1
# n.grad = 1 - (o.data)**2 = 0.5
n.grad = 0.5

# When there's addition, the gradient computes equally
x1w1x2w2.grad  = 0.5
b.grad = 0.5

# Same as previous
x2w2.grad = 0.5
x1w1.grad = 0.5

# For first layer
x1.grad = w1.data * x1w1.grad
w1.grad = x1.data * x1w1.grad
x2.grad = w2.data * x2w2.grad
w2.grad = x2.data * x2w2.grad

draw_dot(o)

- Rather than manually getting gradient, computing each node's gradient using _backward() object

In [65]:
# Inputs
x1 = Value(2.0, label='x1')
x2 = Value(0.0, label='x2')

# Weights
w1 = Value(-3.0, label='w1')
w2 = Value(1.0, label='w2')

# Bias
b = Value(6.88137, label='b')

# Weighted
x1w1 = x1*w1; x1w1.label = 'x1*w1'
x2w2 = x2*w2; x2w2.label = 'x2*w2'

# Weighted sum
x1w1x2w2 = x1w1 + x2w2; x1w1x2w2.label = 'x1*w1 + x2*w2'
n = x1w1x2w2 + b; n.label = 'n'
o = n.tanh(); o.label = 'o'

In [66]:
o.grad = 1
o._backward()
n._backward()
b._backward()
x1w1x2w2._backward()
x1w1._backward()
x2w2._backward()

draw_dot(o)

#### Computing Gradient directly

After writing the backward() object

In [63]:
# Inputs
x1 = Value(2.0, label='x1')
x2 = Value(0.0, label='x2')

# Weights
w1 = Value(-3.0, label='w1')
w2 = Value(1.0, label='w2')

# Bias
b = Value(6.88137, label='b')

# Weighted
x1w1 = x1*w1; x1w1.label = 'x1*w1'
x2w2 = x2*w2; x2w2.label = 'x2*w2'

# Weighted sum
x1w1x2w2 = x1w1 + x2w2; x1w1x2w2.label = 'x1*w1 + x2*w2'
n = x1w1x2w2 + b; n.label = 'n'
o = n.tanh(); o.label = 'o'

In [64]:
o.backward()
draw_dot(o)

#### Neurons

In [167]:
import random

class Neuron:

    def __init__(self, nin):
        self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1,1))

    def __call__(self, x):
        act = sum((wi*xi for wi, xi in zip(self.w, x)),self.b)
        out = act.tanh()
        return out
    
    def parameters(self):
        return self.w + [self.b]

In [168]:
# Without using Layer
x = [2,3]
n = Neuron(2)
n(x)

In [169]:
# Define layer of neurons
class Layer:
    def __init__(self,nin,nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self,x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) ==  1 else outs
    
    def parameters(self):
        return [p for neuron in self.neurons for p in neuron.parameters()]


In [170]:
# Using Layer
x = [2,3]
n = Layer(2,3)
n(x)

In [171]:
class MLP:
    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x    
    
    def parameters(self):
        return [p for neuron in self.layers for p in neuron.parameters()]

In [178]:
x = [-2, 3, 1]
n = MLP(3, [3,3,1])
n(x)


# draw_dot(n(x))

In [179]:
# Example dataset

xs = [
    [2,3,-1],
    [3,-1,0.5],
    [0.5,1,1],
    [1,1,-1]
]

ys = [1,-1,-1,1]

In [180]:
len(n.parameters())

In [83]:
y_pred = [n(x) for x in xs]
y_pred

In [84]:
[(y_out - y_gt)**2 for y_gt, y_out in zip(ys, y_pred)]

In [85]:
loss = sum(((y_out - y_gt)**2 for y_gt, y_out in zip(ys, y_pred)),Value(0.0))
loss

In [86]:
loss.backward()

In [87]:
n.layers[0].neurons[0].w[0].grad

In [88]:
n.layers[0].neurons[0].w[0].data

In [89]:
for p in n.parameters():
    p.data += -0.01 * p.grad

In [90]:
n.layers[0].neurons[0].w[0].data

- Gradient Descent

In [108]:
y_pred = [n(x) for x in xs]
loss = sum(((y_out - y_gt)**2 for y_gt, y_out in zip(ys, y_pred)), Value(0))
loss

In [152]:
loss.backward()

In [153]:
for p in n.parameters():
    p.data += -0.01 * p.grad

In [154]:
y_pred = [n(x) for x in xs]
loss = sum(((y_out - y_gt)**2 for y_gt, y_out in zip(ys, y_pred)), Value(0))
loss

In [155]:
y_pred

In [156]:
n.parameters()

- Making better

In [175]:
# Example dataset
xs = [
    [2,3,-1],
    [3,-1,0.5],
    [0.5,1,1],
    [1,1,-1]
]

ys = [1,-1,-1,1]

In [189]:
for k in range(20):

    # Forward pass
    y_pred = [n(x) for x in xs]
    loss = sum(((y_out - y_gt)**2 for y_gt, y_out in zip(ys, y_pred)), Value(0))


    # Backward pass
    for p in n.parameters():
        p.grad = 0
    loss.backward()

    # Update
    for p in n.parameters():
        p.data += -0.03 * p.grad
    print(k, loss.data)

In [190]:
y_pred